# 🎨 CineAI — Recomendador Inteligente de Filmes e Séries

**Instruções:**
1. Clique no ícone de chave (🔑) na barra lateral esquerda
2. Adicione um secret chamado TMDB_BEARER_TOKEN com seu token da API TMDB
3. Ative o acesso do notebook ao secret
4. Execute todas as células em ordem Ctrl+F9)

In [ ]:
# --- Célula 1: Instalação de Dependências ---
!pip install requests rapidfuzz rich -q

In [ ]:
# --- Célula 2: Configuração do Token ---
import os

# Tenta carregar do Colab Secrets (método seguro)
try:
    from google.colab import userdata
    os.environ["TMDB_BEARER_TOKEN"] = userdata.get("TMDB_BEARER_TOKEN")
    print("✅ Token carregado dos Colab Secrets.")
except Exception:
    # Fallback: cole seu token aqui (menos seguro)
    os.environ["TMDB_BEARER_TOKEN"] = ""  # <-- Cole seu token aqui se não usar Secrets
    if os.environ["TMDB_BEARER_TOKEN"]:
        print("✅ Token carregado manualmente.")
    else:
        print("❌ Token não configurado! Configure via Secrets ou cole na variável acima.")

In [ ]:
# --- Célula 3: CineAI (Código Principal) ---
# -*- coding: utf-8 -*-
import sys, os, re, json, time, random, logging, tempfile, unicodedata, threading
import concurrent.futures as cf
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from rapidfuzz import fuzz, process
from rich.console import Console, Group
from rich.panel import Panel
from rich.prompt import Prompt
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from rich.rule import Rule
from rich.table import Table
from rich.text import Text
from rich.style import Style
from rich.box import ROUNDED
from rich.theme import Theme as RichTheme

# --- Logging ---
log = logging.getLogger("cineai")
logging.basicConfig(level=logging.WARNING)

# --- Tema ---
CINEAI_THEME = RichTheme({
    "primary": "#FF007F", "secondary": "#00B29A", "background": "#1a000d",
    "text": "white", "info": "grey70", "prompt": "cyan",
    "title": "bold #FF007F", "subtitle": "bold #00B29A",
    "success": "bold #00D26A", "warning": "bold #FFD700",
    "error": "bold #FF4F4F", "highlight": "bold #FF007F on #4D0026", "dim": "grey50",
})
# force_terminal=True garante que Rich funcione bem no Colab
console = Console(theme=CINEAI_THEME, force_terminal=True)

# --- Token e API ---
TMDB_BEARER = os.environ.get("TMDB_BEARER_TOKEN", "")
BASE_URL = "https://api.themoviedb.org/3"

# --- Cache ---
CACHE_DIR = Path(tempfile.gettempdir()) / "cineai_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
GENRES_CACHE_FILE = CACHE_DIR / "genres.json"
CATALOG_CACHE_FILE = CACHE_DIR / "catalog.json"
CACHE_EXPIRATION_DAYS = 7

# --- Sessão HTTP com pooling e retry ---
def _build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=0.5, status_forcelist=[500, 502, 503, 504], allowed_methods=["GET"])
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    session.mount("https://", adapter)
    session.headers.update({"Authorization": f"Bearer {TMDB_BEARER}", "Accept": "application/json"})
    return session

SESSION = _build_session()

# --- Rate limiter global thread-safe ---
_rate_lock = threading.Lock()
_last_request_time = 0.0
_MIN_REQUEST_INTERVAL = 0.05

def _rate_limited_wait():
    global _last_request_time
    with _rate_lock:
        now = time.monotonic()
        elapsed = now - _last_request_time
        if elapsed < _MIN_REQUEST_INTERVAL:
            time.sleep(_MIN_REQUEST_INTERVAL - elapsed)
        _last_request_time = time.monotonic()

# --- Funções Utilitárias ---
def _norm(s: Optional[str]) -> str:
    if not s: return ""
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)).lower() if not unicodedata.combining(c)).strip()

def _get_year(date_str: Optional[str]) -> Optional[int]:
    if not date_str: return None
    try:
        year = int(date_str.split("-")[0])
        return year if 1888 <= year <= 2030 else None
    except (ValueError, IndexError): return None

def _safe_float(value: Any, default: float = 0.0) -> float:
    try:
        result = float(value)
        return default if result != result else result  # NaN check
    except (TypeError, ValueError): return default

def _safe_int(value: Any, default: int = 0) -> int:
    try: return int(value)
    except (TypeError, ValueError): return default

def _is_nenhum(s: str) -> bool:
    return not s or _norm(s) in ("nenhum", "nenhuma", "nada", "n/a", "")

# --- Cache (leitura/escrita atômica) ---
def _read_cache(filepath: Path) -> Optional[Any]:
    try:
        age_days = (time.time() - filepath.stat().st_mtime) / 86400
        if age_days > CACHE_EXPIRATION_DAYS: return None
        with open(filepath, "r", encoding="utf-8") as f: return json.load(f)
    except FileNotFoundError: return None
    except (json.JSONDecodeError, OSError) as exc:
        log.warning("Cache corrompido em %s: %s", filepath, exc); return None

def _write_cache(filepath: Path, data: Any) -> None:
    try:
        tmp_fd, tmp_path = tempfile.mkstemp(dir=str(filepath.parent), suffix=".tmp")
        try:
            with os.fdopen(tmp_fd, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            os.replace(tmp_path, str(filepath))
        except BaseException:
            try: os.unlink(tmp_path)
            except OSError: pass
            raise
    except OSError as exc: log.warning("Falha ao escrever cache %s: %s", filepath, exc)

# --- API TMDB ---
def _tmdb_request(path: str, params: Optional[Dict] = None, retries: int = 3) -> Dict:
    wait_time = 0.3; params = params or {}
    for attempt in range(retries + 1):
        _rate_limited_wait()
        try:
            r = SESSION.get(f"{BASE_URL}{path}", params=params, timeout=20)
            if r.status_code == 429:
                time.sleep(min(10.0, _safe_float(r.headers.get("Retry-After"), wait_time)))
                wait_time = min(5.0, wait_time * 2); continue
            r.raise_for_status(); return r.json()
        except requests.RequestException as exc:
            if attempt < retries:
                time.sleep(wait_time + random.uniform(0, wait_time * 0.3))
                wait_time = min(5.0, wait_time * 2)
            else: log.warning("Falha após %d tentativas em %s: %s", retries + 1, path, exc)
    return {}

# --- Gêneros ---
_genres_cache: Optional[Dict] = None

def _get_genres_cached() -> Dict[str, Dict[str, str]]:
    global _genres_cache
    if _genres_cache: return _genres_cache
    cached = _read_cache(GENRES_CACHE_FILE)
    if cached and "movie_genres" in cached and "tv_genres" in cached:
        _genres_cache = cached; return _genres_cache
    mg = _tmdb_request("/genre/movie/list", {"language": "pt-BR"}).get("genres", [])
    tg = _tmdb_request("/genre/tv/list", {"language": "pt-BR"}).get("genres", [])
    genres = {
        "movie_genres": {str(g["id"]): g["name"] for g in mg if "id" in g and "name" in g},
        "tv_genres": {str(g["id"]): g["name"] for g in tg if "id" in g and "name" in g},
    }
    _write_cache(GENRES_CACHE_FILE, genres); _genres_cache = genres; return genres

# --- Construção de Catálogo ---
def _build_catalog_item(it: Dict, kind: str, g_map: Dict[str, str]) -> Optional[Dict]:
    item_id = it.get("id")
    if not item_id: return None
    tk = "title" if kind == "movie" else "name"
    otk = "original_title" if kind == "movie" else "original_name"
    dk = "release_date" if kind == "movie" else "first_air_date"
    title = it.get(tk) or it.get(otk) or ""
    if not title: return None
    g_list = [g_map[str(gid)] for gid in it.get("genre_ids", []) if str(gid) in g_map]
    return {
        "id": item_id, "type": kind, "title": title, "genres": "|".join(g_list),
        "year": _get_year(it.get(dk)), "runtime": None,
        "vote_avg": _safe_float(it.get("vote_average")),
        "vote_cnt": _safe_int(it.get("vote_count")),
        "popularity": _safe_float(it.get("popularity")),
    }

def _discover_page(task: tuple) -> Tuple[List[Dict], bool]:
    kind, g, sort_by, date_range, page, lang, all_g, keywords = task
    params = {"language": lang, "sort_by": sort_by, "page": page, "include_adult": "false", "vote_count.gte": 100}
    if keywords: params["with_keywords"] = keywords
    data = _tmdb_request(f"/discover/{kind}", params)
    g_map = all_g["movie_genres"] if kind == "movie" else all_g["tv_genres"]
    results = [item for it in data.get("results", []) if (item := _build_catalog_item(it, kind, g_map))]
    return results, data.get("page", 0) < data.get("total_pages", 0)

def _fetch_live_catalog(target: int, lang: str, workers: int) -> List[Dict]:
    g = _get_genres_cached()
    m_ids_map = {name: gid for gid, name in g["movie_genres"].items()}
    t_ids_map = {name: gid for gid, name in g["tv_genres"].items()}
    m_ids = list(m_ids_map.values()); t_ids = list(t_ids_map.values())

    series_priority_tasks = []
    anime_keyword = "210024"
    anime_genres = [t_ids_map[gn] for gn in ["Animação", "Ação e Aventura", "Sci-Fi & Fantasy"] if gn in t_ids_map]
    if anime_genres:
        for sort_key in ["popularity.desc", "vote_average.desc"]:
            for page in range(1, 11):
                series_priority_tasks.append(("tv", anime_genres, sort_key, ("2000-01-01", "2025-12-31"), page, lang, g, anime_keyword))
    for sort_key in ["popularity.desc", "vote_average.desc"]:
        for genre_chunk in [t_ids[i:i+2] for i in range(0, len(t_ids), 2)]:
            for page in range(1, 4):
                series_priority_tasks.append(("tv", genre_chunk, sort_key, ("2010-01-01", "2025-12-31"), page, lang, g, None))

    general_tasks = []
    decades = [("1990-01-01","1999-12-31"),("2000-01-01","2009-12-31"),("2010-01-01","2019-12-31"),("2020-01-01","2025-12-31")]
    sorts = ["vote_average.desc", "popularity.desc"]
    for d in decades:
        for s_ in sorts:
            for chunk in [random.sample(m_ids, len(m_ids))[i:i+2] for i in range(0, len(m_ids), 2)]:
                general_tasks.append(("movie", chunk, s_, d, 1, lang, g, None))
            for chunk in [random.sample(t_ids, len(t_ids))[i:i+2] for i in range(0, len(t_ids), 2)]:
                general_tasks.append(("tv", chunk, s_, d, 1, lang, g, None))

    random.shuffle(series_priority_tasks); random.shuffle(general_tasks)
    tasks = deque(series_priority_tasks + general_tasks)
    catalog: List[Dict] = []; seen_ids: set = set()

    with Progress(SpinnerColumn(style="primary"), TextColumn("[progress.description]{task.description}"),
                  BarColumn(bar_width=None), TextColumn("[progress.percentage]{task.percentage:>3.0f}%"),
                  TextColumn("• {task.completed}/{task.total} títulos"), TimeElapsedColumn(),
                  console=console, transient=True) as prog:
        task_id = prog.add_task("[dim]Buscando na API TMDB...[/]", total=target)
        with cf.ThreadPoolExecutor(max_workers=workers) as ex:
            futures: Dict[cf.Future, tuple] = {}
            for _ in range(min(len(tasks), workers * 2)):
                if tasks:
                    t = tasks.popleft(); futures[ex.submit(_discover_page, t)] = t
            while futures and len(catalog) < target:
                done, _ = cf.wait(futures, return_when=cf.FIRST_COMPLETED)
                for fut in done:
                    orig_task = futures.pop(fut)
                    try:
                        batch, has_more = fut.result()
                        added = 0
                        for item in batch:
                            key = (item["type"], item["id"])
                            if key not in seen_ids: seen_ids.add(key); catalog.append(item); added += 1
                        prog.update(task_id, advance=added)
                        kind, gn, s_, d, p, ln, gs, kw = orig_task
                        if has_more and added > 0 and p < 15 and len(catalog) < target:
                            tasks.append((kind, gn, s_, d, p + 1, ln, gs, kw))
                    except Exception as exc: log.debug("Erro ao processar página: %s", exc)
                    if tasks and len(futures) < workers * 2:
                        t = tasks.popleft(); futures[ex.submit(_discover_page, t)] = t
    return catalog[:target]

def build_catalog_cached(target: int, force_rebuild: bool) -> List[Dict]:
    if not force_rebuild:
        cached = _read_cache(CATALOG_CACHE_FILE)
        if cached and isinstance(cached, list) and len(cached) > 0:
            console.print(f"[success]✓[/] Catálogo pronto com {len(cached)} títulos.")
            return cached
    console.print("[warning]![/] Cache inválido. Buscando da API (pode levar alguns minutos)...")
    catalog = _fetch_live_catalog(target, "pt-BR", 10)
    _write_cache(CATALOG_CACHE_FILE, catalog)
    console.print(f"[success]✓[/] Catálogo pronto com {len(catalog)} títulos.")
    return catalog

# --- Busca Específica ---
def _get_search_id(name: str, path: str) -> Optional[str]:
    if _is_nenhum(name): return None
    data = _tmdb_request(path, {"language": "pt-BR", "query": name})
    results = data.get("results", [])
    if results: return str(results[0]["id"])
    console.print(f"[warning]Aviso: Termo '{name}' não encontrado em {path}.[/]")
    return None

def _get_keyword_ids(terms: List[str]) -> Optional[str]:
    ids = []
    for term in terms:
        if _is_nenhum(term): continue
        data = _tmdb_request("/search/keyword", {"query": term})
        results = data.get("results", [])
        if results: ids.append(str(results[0]["id"]))
        else: console.print(f"[warning]Aviso: Palavra-chave '{term}' não encontrada.[/]")
    return ",".join(ids) if ids else None

def _build_discover_params(prefs: Dict[str, Any]) -> Dict[str, Any]:
    g = _get_genres_cached()
    m_ids_map = {name: gid for gid, name in g["movie_genres"].items()}
    t_ids_map = {name: gid for gid, name in g["tv_genres"].items()}
    inc_ids_m = [str(m_ids_map[n]) for n in prefs["inc"] if n in m_ids_map]
    inc_ids_t = [str(t_ids_map[n]) for n in prefs["inc"] if n in t_ids_map]
    exc_ids_m = [str(m_ids_map[n]) for n in prefs["exc"] if n in m_ids_map]
    exc_ids_t = [str(t_ids_map[n]) for n in prefs["exc"] if n in t_ids_map]
    if prefs["type"] == "movie": inc_g, exc_g = inc_ids_m, exc_ids_m
    elif prefs["type"] == "tv": inc_g, exc_g = inc_ids_t, exc_ids_t
    else: inc_g, exc_g = list(set(inc_ids_m + inc_ids_t)), list(set(exc_ids_m + exc_ids_t))
    params: Dict[str, Any] = {"language": "pt-BR", "include_adult": "false", "vote_count.gte": 150}
    if inc_g: params["with_genres"] = ",".join(inc_g)
    if exc_g: params["without_genres"] = ",".join(exc_g)
    with console.status("[dim]Traduzindo termos da API...[/]", spinner="dots"):
        if prefs.get("keywords_raw"):
            kw_ids = _get_keyword_ids(prefs["keywords_raw"])
            if kw_ids: params["with_keywords"] = kw_ids
        if prefs.get("actor_raw") and not _is_nenhum(prefs["actor_raw"]):
            pid = _get_search_id(prefs["actor_raw"], "/search/person")
            if pid: params["with_cast"] = pid
        if prefs.get("director_raw") and not _is_nenhum(prefs["director_raw"]):
            pid = _get_search_id(prefs["director_raw"], "/search/person")
            if pid: params["with_crew"] = pid
        if prefs.get("company_raw") and not _is_nenhum(prefs["company_raw"]):
            cid = _get_search_id(prefs["company_raw"], "/search/company")
            if cid: params["with_companies"] = cid
        if prefs.get("network_raw") and not _is_nenhum(prefs["network_raw"]):
            nid = _get_search_id(prefs["network_raw"], "/search/network")
            if nid: params["with_networks"] = nid
        if prefs.get("year_raw"):
            key = "primary_release_year" if prefs["type"] != "tv" else "first_air_date_year"
            params[key] = prefs["year_raw"]
        if prefs.get("min_vote_raw"): params["vote_average.gte"] = prefs["min_vote_raw"]
        if prefs.get("rating_raw") and prefs["rating_raw"] != "NENHUM":
            params["certification_country"] = "BR"; params["certification.lte"] = prefs["rating_raw"]
    if "primary_release_year" not in params and "first_air_date_year" not in params:
        if prefs.get("classic_focus"):
            dk = "primary_release_date.lte" if prefs["type"] != "tv" else "first_air_date.lte"
            params[dk] = "2000-01-01"; params["sort_by"] = "vote_average.desc"
        elif prefs.get("prefer_new"):
            dk = "primary_release_date.gte" if prefs["type"] != "tv" else "first_air_date.gte"
            params[dk] = "2010-01-01"; params["sort_by"] = "popularity.desc"
    if "sort_by" not in params:
        params["sort_by"] = "vote_average.desc" if prefs["w_rating"] > prefs["w_pop"] else "popularity.desc"
    return params

def _fetch_live_discover(kind: str, params: Dict[str, Any], target_pages: int = 3) -> List[Dict]:
    catalog: List[Dict] = []; seen_ids: set = set()
    g_map = _get_genres_cached()["movie_genres"] if kind == "movie" else _get_genres_cached()["tv_genres"]
    with Progress(SpinnerColumn(style="primary"), TextColumn("[progress.description]{task.description}"),
                  BarColumn(), TextColumn("{task.completed}/{task.total} págs"), console=console, transient=True) as prog:
        tid = prog.add_task(f"[dim]Buscando '{kind}' ao vivo...[/]", total=target_pages)
        for p in range(1, target_pages + 1):
            params["page"] = p
            data = _tmdb_request(f"/discover/{kind}", params)
            results = data.get("results", [])
            if not results: break
            for it in results:
                item = _build_catalog_item(it, kind, g_map)
                if item:
                    key = (kind, item["id"])
                    if key not in seen_ids: seen_ids.add(key); catalog.append(item)
            prog.update(tid, advance=1)
    return catalog

# --- Pontuação ---
GEN_CANON = ["Ação","Aventura","Animação","Comédia","Crime","Documentário","Drama","Família",
             "Fantasia","História","Terror","Música","Mistério","Romance","Ficção Científica",
             "Cinema TV","Thriller","Guerra","Faroeste"]
THEMES = {
    "heist":["Crime","Ação"],"espacial":["Ficção Científica","Aventura"],
    "medieval":["Fantasia","História"],"super-heroi":["Ação","Aventura","Fantasia"],
    "politica":["Drama","Thriller"],"biografia":["Drama","História"],
    "noir":["Crime","Mistério"],"musical":["Música","Romance"],"suspense":["Thriller","Mistério"],
}

def map_terms_to_genres(terms: List[str]) -> List[str]:
    out: set = set(); norm_canon = [_norm(g) for g in GEN_CANON]
    for term in terms:
        if _is_nenhum(term): continue
        nt = _norm(term)
        for tk, tg in THEMES.items():
            if fuzz.partial_ratio(nt, tk) > 85: out.update(tg)
        best = process.extractOne(nt, norm_canon, scorer=fuzz.token_set_ratio)
        if best and best[1] > 75: out.add(GEN_CANON[best[2]])
    return list(out)

def get_duration_score(pref: Optional[str], runtime: Optional[int]) -> float:
    if runtime is None: return 0.5
    t = _norm(pref or "")
    if "curt" in t: return max(0.0, 1 - abs(runtime - 90) / 90)
    if "medi" in t: return max(0.0, 1 - abs(runtime - 120) / 60)
    if "long" in t: return max(0.0, 1 - abs(runtime - 180) / 180)
    return 0.5

TYPE_MATCH_STRONG_BONUS = 60

def score_item(item: Dict, prefs: Dict[str, Any]) -> float:
    s = 0.0; i_g_norm = _norm(item["genres"])
    if prefs["type"] and prefs["type"] == item["type"]: s += TYPE_MATCH_STRONG_BONUS
    for g_norm in prefs["inc_norm"]:
        if g_norm in i_g_norm: s += 15
    for g_norm in prefs["exc_norm"]:
        if g_norm in i_g_norm: s -= 25
    s += 10 * get_duration_score(prefs["dur"], item.get("runtime"))
    va = _safe_float(item.get("vote_avg")); vc = _safe_int(item.get("vote_cnt")); pop = _safe_float(item.get("popularity"))
    q = (va / 10.0) * min(1.0, vc / 5000.0); p = min(1.0, pop / 1000.0)
    s += prefs["w_rating"] * q * 12; s += prefs["w_pop"] * p * 12
    y = item.get("year") or 2000
    if prefs.get("classic_focus"): s += max(0, (2000 - y) / 10)
    elif prefs["prefer_new"]: s += max(0, (y - 2000) / 3)
    else: s -= max(0, (y - 2010) / 4)
    return s

# --- Detalhes ---
def _parse_certification(data: Dict, kind: str) -> str:
    try:
        if kind == "movie":
            for r in data.get("release_dates", {}).get("results", []):
                if r.get("iso_3166_1") == "BR":
                    for cert in r.get("release_dates", []):
                        if c := cert.get("certification"): return c
        else:
            for r in data.get("content_ratings", {}).get("results", []):
                if r.get("iso_3166_1") == "BR": return r.get("rating", "N/A")
    except (KeyError, TypeError): pass
    return "N/A"

def _fetch_details_concurrent(items: List[Dict]) -> List[Dict]:
    def fetch_one(item: Dict) -> Dict:
        kind = item["type"]
        ap = "credits,watch/providers,keywords,recommendations"
        ap += ",release_dates" if kind == "movie" else ",content_ratings"
        data = _tmdb_request(f"/{kind}/{item['id']}", {"language": "pt-BR", "append_to_response": ap})
        if not data: return item
        if kind == "movie": item["runtime"] = _safe_int(data.get("runtime")) or None
        else:
            ep_rt = data.get("episode_run_time", [])
            item["runtime"] = int(sum(ep_rt) / len(ep_rt)) if ep_rt else None
        item["synopsis"] = data.get("overview") or "Sinopse não disponível."
        if kind == "movie":
            dirs = [c["name"] for c in data.get("credits", {}).get("crew", []) if c.get("job") == "Director"]
            item["director"] = ", ".join(dirs[:2]) if dirs else "N/A"
        else:
            creators = [c["name"] for c in data.get("created_by", [])]
            item["director"] = ", ".join(creators[:2]) if creators else "N/A"
        cast = [c["name"] for c in data.get("credits", {}).get("cast", [])[:4]]
        item["cast"] = ", ".join(cast) if cast else "N/A"
        prov_data = data.get("watch/providers", {}).get("results", {}).get("BR", {})
        streaming = [p["provider_name"] for p in prov_data.get("flatrate", [])]
        item["providers"] = ", ".join(list(dict.fromkeys(streaming))) if streaming else "Não encontrado em streaming (assinatura)"
        kw_data = data.get("keywords", {}); kw_list = kw_data.get("keywords", kw_data.get("results", []))
        item["keywords"] = ", ".join([k["name"] for k in kw_list[:5]]) if kw_list else "N/A"
        rec_data = data.get("recommendations", {}).get("results", [])
        if rec_data:
            rn = [r.get("title") or r.get("name") for r in rec_data[:3] if r.get("title") or r.get("name")]
            item["recommendations"] = ", ".join(rn)
        else: item["recommendations"] = "Nenhuma recomendação similar encontrada."
        item["tagline"] = data.get("tagline") or ""; item["rating_br"] = _parse_certification(data, kind)
        item["companies"] = ", ".join([c["name"] for c in data.get("production_companies", [])[:2]])
        if kind == "tv": item["seasons"] = data.get("number_of_seasons"); item["episodes"] = data.get("number_of_episodes")
        return item
    with cf.ThreadPoolExecutor(max_workers=4) as ex: return list(ex.map(fetch_one, items))

# --- Comentários ---
COMMENT_TEMPLATES = {
    "Ação": ["...pura octanagem."], "Aventura": ["...jornada épica."],
    "Comédia": ["Gargalhadas garantidas..."], "Drama": ["...história emocionante."],
    "Ficção Científica": ["...expande a mente."], "Terror": ["...luzes acesas."],
    "Romance": ["...para aquecer o coração."], "Mistério": ["...quebra-cabeça."],
    "Animação": ["...deslumbrante."], "default": ["...merece uma chance."],
}
QUALIFIER_PHRASES = {
    "acclaimed": ["Aclamado pela crítica..."], "popular": ["O título do momento..."],
    "underrated": ["Uma joia subestimada..."], "classic": ["Um verdadeiro clássico..."],
}

def generate_ai_comment_local(item: Dict) -> str:
    genres = item.get("genres", "").split("|")
    pg = genres[0] if genres and genres[0] else "default"
    parts = [random.choice(COMMENT_TEMPLATES.get(pg, COMMENT_TEMPLATES["default"]))]
    va = _safe_float(item.get("vote_avg")); yr = item.get("year") or 2025; pop = _safe_float(item.get("popularity"))
    if va >= 8.2: parts.append(random.choice(QUALIFIER_PHRASES["acclaimed"]))
    elif yr < 2000: parts.append(random.choice(QUALIFIER_PHRASES["classic"]))
    elif pop > 1000: parts.append(random.choice(QUALIFIER_PHRASES["popular"]))
    elif 6.8 <= va < 7.8: parts.append(random.choice(QUALIFIER_PHRASES["underrated"]))
    return " ".join(parts)

# --- UI ---
def banner():
    console.print(Panel(Text("🎨 CineAI", style="highlight", justify="center"), box=ROUNDED, padding=(1, 0), border_style="primary"))
    console.print(Text("As melhores sugestões para você!", style="info", justify="center"))

def get_match_color(score_pct: float) -> Style:
    if score_pct > 92: return Style(color="#00FF7F")
    if score_pct > 80: return Style(color="#ADFF2F")
    if score_pct > 65: return "warning"
    return Style(color="#FFA07A")

def show_results(scored_items: List[Tuple[float, Dict]], top_n: int = 3, is_live_search: bool = False):
    if not scored_items:
        console.print("\n[warning]Nenhum resultado encontrado[/] para essa combinação de filtros.")
        return
    scores = [s for s, _ in scored_items[:top_n]]; min_s = min(scores); max_s = max(scores); sr = max_s - min_s
    title_text = "Top 3 (Busca Específica API)" if is_live_search else "Resumo (Busca Local)"
    st = Table(title=Text(title_text, style="title"), header_style="secondary", box=ROUNDED, border_style="dim")
    st.add_column("Compatibilidade", justify="center", width=15); st.add_column("Tipo", width=5)
    st.add_column("Título", min_width=20, overflow="fold"); st.add_column("Ano", width=4)
    st.add_column("Gêneros", overflow="fold"); st.add_column("🕒", justify="center", width=7)
    st.add_column("⭐", justify="center", width=5)
    for idx, (score, item) in enumerate(scored_items[:top_n]):
        if is_live_search: pt = Text(f"#{idx+1} API", style=get_match_color(95))
        else:
            pct = max(0.0, min(100.0, 40 + (score - min_s) / (sr if sr > 0 else 1) * 59))
            pt = Text(f"{pct:.1f}%", style=get_match_color(pct))
        ts = "success" if item["type"]=="movie" else "secondary"
        tl = f"[bold {ts}]{'FILME' if item['type']=='movie' else 'SÉRIE'}[/]"
        rt = f"{item.get('runtime')}m" if item.get('runtime') else "[dim]N/A[/]"
        ns = get_match_color(item['vote_avg'] * 10)
        st.add_row(pt, tl, Text(item["title"], style="text"), str(item.get("year","")),
                   Text(item["genres"].replace("|",", "), style="dim"), rt, Text(f"{item['vote_avg']:.1f}", style=ns))
    console.print(st); console.print(Rule(style="dim", characters="·"))
    for i, (_, item) in enumerate(scored_items[:top_n], 1):
        title = Text.from_markup(f"[bold]#{i} {item['title']}[/] [dim]({item.get('year','N/A')})[/]")
        if tl := item.get("tagline"): title.append(Text(f'\n \\"{tl}\\"', style="warning italic"))
        syn_str = item.get('synopsis', 'Sinopse não disponível.')
        syn_st = "dim" if "não disponível" in syn_str else "text"
        synopsis = Text(f"\n  {syn_str}\n", style=syn_st, justify="full")
        dg = Table.grid(padding=(0, 1)); dg.add_column(width=16); dg.add_column()
        dl = 'Diretor:' if item['type'] == 'movie' else 'Criador:'
        def _dr(label, value, na=("N/A",)):
            sty = "dim" if any(m in value for m in na) else "text"
            dg.add_row(f"[prompt]{label}[/]", Text(value, style=sty))
        _dr("Onde Ver (BR):", item.get('providers','N/A'), ("Não encontrado",))
        _dr(dl, item.get('director','N/A')); _dr("Elenco:", item.get('cast','N/A'))
        _dr("Classificação:", item.get('rating_br','N/A'))
        if item['type'] == 'tv':
            dg.add_row("[prompt]Temporadas:[/]", Text(f"{item.get('seasons','N/A')} ({item.get('episodes','N/A')} eps)", style="text"))
        _dr("Produtora(s):", item.get('companies','N/A')); _dr("Palavras-chave:", item.get('keywords','N/A'))
        _dr("Similares:", item.get('recommendations','N/A'), ("Nenhuma",))
        ai = Text.from_markup(f"\n[primary b]💬 Comentário do CineAI:[/]\n[i]{item.get('ai_comment','...')}[/i]")
        console.print(Panel(Group(synopsis, dg, ai), title=title, title_align="left", box=ROUNDED, border_style="primary", padding=(1,1)))

# --- Prompt fuzzy com limite de tentativas ---
MAX_PROMPT_ATTEMPTS = 10

def fuzzy_prompt(prompt_text: str, choices: List[str], default: Optional[str] = None,
                 threshold: int = 70, explanation: str = "") -> str:
    prompt_render = Text.from_markup(f"[prompt]❯ {prompt_text}[/] [info]({explanation})[/]")
    for attempt in range(MAX_PROMPT_ATTEMPTS):
        user_input = Prompt.ask(prompt_render, default=default, show_default=True)
        if not user_input: user_input = default
        if not user_input:
            console.print("  [error]Resposta não pode ser vazia.[/error]"); continue
        norm_input = _norm(user_input); norm_choices = [_norm(c) for c in choices]
        best = process.extractOne(norm_input, norm_choices, scorer=fuzz.token_set_ratio)
        if best and best[1] >= threshold: return choices[best[2]]
        remaining = MAX_PROMPT_ATTEMPTS - attempt - 1
        console.print(f"  [error]Opção '[white]{user_input}[/white]' não reconhecida. ({remaining} restantes)[/error]")
    fallback = default if default and default in choices else choices[0]
    console.print(f"  [warning]Usando padrão: {fallback}[/warning]")
    return fallback

# --- Painéis de Preferências ---
def ask_preferences_panel() -> Dict[str, Any]:
    console.print(Panel(Text("Busca Normal", style="title", justify="center"), subtitle="Baseada no Cache Local", box=ROUNDED, border_style="secondary"))
    console.print(Text("Responda às perguntas para calibrar as recomendações.", style="info", justify="center"))
    console.print("")
    tipo_map = {"Filme":"movie","Série":"tv","Tanto faz":""}
    foco_map = {"Nota":(1.0,0.4),"Popularidade":(0.4,1.0),"Equilíbrio":(0.8,0.8)}
    tipo = fuzzy_prompt("Qual o formato?", list(tipo_map.keys()), "Tanto faz", explanation="[b]F[/b]ilme / [b]S[/b]érie / [b]T[/b]anto faz")
    foco = fuzzy_prompt("Qual o seu foco?", list(foco_map.keys()), "Equilíbrio", explanation="[b]N[/b]ota / [b]P[/b]opularidade / [b]E[/b]quilíbrio")
    pr = Text.from_markup("[prompt]❯ Gêneros ou temas que você curte?[/] [info](Separe por vírgulas)[/]")
    inc_raw = Prompt.ask(pr, default="Nenhum")
    pr = Text.from_markup("[prompt]❯ Gêneros ou temas para evitar?[/] [info](Separe por vírgulas)[/]")
    exc_raw = Prompt.ask(pr, default="Nenhum")
    dur = fuzzy_prompt("Qual a duração ideal?", ["Curta","Média","Longa","Qualquer"], "Qualquer", explanation="[b]C[/b]urta / [b]M[/b]édia / [b]L[/b]onga / [b]Q[/b]ualquer")
    pn = fuzzy_prompt("Preferência por títulos recentes (pós 2010)?", ["Sim","Não"], "Não", explanation="[b]S[/b]im / [b]N[/b]ão")
    w_r, w_p = foco_map[foco]
    inc_g = map_terms_to_genres(re.split(r"[,;\\|/]+", inc_raw))
    exc_g = map_terms_to_genres(re.split(r"[,;\\|/]+", exc_raw))
    console.print(f"\n[dim]Gêneros mapeados: Incluindo [warning]{inc_g or 'Nenhum'}[/] | Excluindo [error]{exc_g or 'Nenhum'}[/dim]")
    return {"type":tipo_map[tipo],"inc":inc_g,"exc":exc_g,"inc_norm":[_norm(g) for g in inc_g],
            "exc_norm":[_norm(g) for g in exc_g],"dur":dur,"prefer_new":pn=="Sim","w_rating":w_r,"w_pop":w_p,"is_specific":False}

def ask_specific_search_panel() -> Dict[str, Any]:
    console.print(Panel(Text("Busca Específica", style="title", justify="center"), subtitle="Direto da API", box=ROUNDED, border_style="primary"))
    console.print(Text("Preencha o máximo de campos para busca precisa.", style="info", justify="center"))
    console.print("")
    tipo_map = {"Filme":"movie","Série":"tv","Tanto faz":""}
    foco_map = {"Nota":(1.0,0.4),"Popularidade":(0.4,1.0),"Equilíbrio":(0.8,0.8)}
    tipo = fuzzy_prompt("Qual o formato?", list(tipo_map.keys()), "Tanto faz", explanation="[b]F[/b]ilme / [b]S[/b]érie / [b]T[/b]anto faz")
    foco = fuzzy_prompt("Qual o seu foco?", list(foco_map.keys()), "Equilíbrio", explanation="[b]N[/b]ota / [b]P[/b]opularidade / [b]E[/b]quilíbrio")
    pr = Text.from_markup("[prompt]❯ Gêneros ou temas que você curte?[/] [info](Separe por vírgulas)[/]")
    inc_raw = Prompt.ask(pr, default="Nenhum")
    pr = Text.from_markup("[prompt]❯ Gêneros ou temas para evitar?[/] [info](Separe por vírgulas)[/]")
    exc_raw = Prompt.ask(pr, default="Nenhum")
    pn = fuzzy_prompt("Preferência por títulos recentes (pós 2010)?", ["Sim","Não"], "Não", explanation="[b]S[/b]im / [b]N[/b]ão")
    console.print(Rule(style="dim", characters="·"))
    console.print(Text("Filtros Específicos", style='subtitle', justify="center"))
    def _af(label, hint, default="Nenhum"):
        return Prompt.ask(Text.from_markup(f"[prompt]❯ {label}[/] [info]({hint})[/]"), default=default)
    year_raw = _af("Ano de lançamento específico?", "Ex: 1999", "")
    min_vote_raw = _af("Nota mínima (0-10)?", "Ex: 8.5", "")
    rating_raw = _af("Classificação indicativa (BR) máxima?", "Ex: L, 10, 12, 14, 16, 18").upper()
    keywords_raw = _af("Tema ou palavra-chave específica?", "Ex: cyberpunk, viagem no tempo")
    actor_raw = _af("Incluir algum ator ou atriz?", "Ex: Tom Hanks")
    director_raw = _af("Incluir algum diretor(a)?", "Ex: Christopher Nolan")
    company_raw = _af("De qual Produtora ou Estúdio?", "Ex: A24, Marvel")
    network_raw = _af("De qual Rede ou Streaming?", "Ex: HBO, Netflix")
    w_r, w_p = foco_map[foco]
    inc_g = map_terms_to_genres(re.split(r"[,;\\|/]+", inc_raw))
    exc_g = map_terms_to_genres(re.split(r"[,;\\|/]+", exc_raw))
    console.print(f"\n[dim]Gêneros mapeados: Incluindo [warning]{inc_g or 'Nenhum'}[/] | Excluindo [error]{exc_g or 'Nenhum'}[/dim]")
    return {"type":tipo_map[tipo],"inc":inc_g,"exc":exc_g,"inc_norm":[_norm(g) for g in inc_g],
            "exc_norm":[_norm(g) for g in exc_g],"dur":"Qualquer","prefer_new":pn=="Sim","w_rating":w_r,"w_pop":w_p,
            "keywords_raw":[k.strip() for k in re.split(r"[,;\\|/]+", keywords_raw) if k.strip() and not _is_nenhum(k)],
            "actor_raw":actor_raw.strip(),"director_raw":director_raw.strip(),
            "company_raw":company_raw.strip(),"network_raw":network_raw.strip(),
            "year_raw":year_raw.strip(),"min_vote_raw":min_vote_raw.strip(),
            "rating_raw":rating_raw.strip(),"is_specific":True}

print("✅ CineAI carregado com sucesso!")

In [ ]:
# --- Célula 4: Executar o CineAI ---
# Configurações (altere se quiser)
TARGET_TITULOS = 2500    # Quantidade de títulos no catálogo
FORCAR_REBUILD = False   # True para forçar reconstruir o cache

# --- Validação do Token ---
if not TMDB_BEARER:
    console.print("[error]❌ Token TMDB não configurado![/]")
    console.print("[info]Configure na Célula 2 via Colab Secrets ou colando manualmente.[/]")
else:
    # --- Execução Principal ---
    banner()
    try:
        _get_genres_cached()
    except Exception as e:
        console.print(f"[error]ERRO CRÍTICO: Falha ao carregar gêneros. Detalhe: {e}[/error]")
        raise SystemExit(1)

    catalog_base = build_catalog_cached(TARGET_TITULOS, FORCAR_REBUILD)
    if not catalog_base:
        console.print(Panel("[error]Falha ao construir o catálogo base.[/]", box=ROUNDED, border_style='error'))
        raise SystemExit(1)

    console.print(Rule(style="dim"))
    prefs = ask_preferences_panel()

    while True:
        scored_items: List[Tuple[float, Dict]] = []
        is_live_search = prefs.get("is_specific", False)

        if is_live_search:
            console.print("\n[primary b]![/] Busca específica ativada. Consultando a API ao vivo...")
            api_params = _build_discover_params(prefs)
            live_catalog: List[Dict] = []
            if prefs["type"] == "movie": live_catalog = _fetch_live_discover("movie", api_params.copy())
            elif prefs["type"] == "tv": live_catalog = _fetch_live_discover("tv", api_params.copy())
            else:
                live_catalog.extend(_fetch_live_discover("movie", api_params.copy(), target_pages=2))
                live_catalog.extend(_fetch_live_discover("tv", api_params.copy(), target_pages=2))
            scored_items = [(100 - i, item) for i, item in enumerate(live_catalog)]
        else:
            console.print(f"\n[dim]Buscando no catálogo local de {len(catalog_base)} títulos...[/]")
            with console.status("[dim]🧠 Calculando recomendações...[/]", spinner="dots"):
                scored_items = sorted([(score_item(item, prefs), item) for item in catalog_base], key=lambda x: x[0], reverse=True)

        top_items = [item for _, item in scored_items[:3]]
        if top_items:
            with console.status("[dim]Buscando detalhes completos...[/]", spinner="dots"):
                _fetch_details_concurrent(top_items)
            for item in top_items:
                item['ai_comment'] = generate_ai_comment_local(item)

        console.print(Rule(style="dim", characters=" "))
        show_results(scored_items, top_n=3, is_live_search=is_live_search)
        console.print("")

        action = fuzzy_prompt("O que fazer agora?",
                              ["Busca Específica (API)", "Nova Busca Normal", "Sair"],
                              default="Nova Busca Normal",
                              explanation="[b]A[/b]PI (Específica) / [b]N[/b]ormal (Rápida) / [b]S[/b]air")

        if action == "Sair": break
        elif action == "Nova Busca Normal":
            console.print(Rule(style="dim")); prefs = ask_preferences_panel()
        elif action == "Busca Específica (API)":
            console.print(Rule(style="dim")); prefs = ask_specific_search_panel()

    console.print(Panel(Text("🍿 Bom filme/série!", style="highlight"), box=ROUNDED, padding=(1, 4), border_style="primary"))